In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [6]:
import numpy as np
import pandas as pd

from src.preprocessing.loader import DataLoader
from src.preprocessing.cleaner import DataCleaner
from src.preprocessing.encoder import DataEncoder
from src.preprocessing.scaler import DataScaler
from src.preprocessing.splitter import DataSplitter

from src.config.one_class_svm_config import OneClassSVMConfig
from src.models.one_class_svm import OneClassSVMModel

In [7]:
DATASET = "../datasets/processed/cicids2017.csv"

In [8]:
from src.preprocessing.loader import DataLoader

loader = DataLoader()

files = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
]

df = loader.load_multiple(files)

print(df.shape)

2026-07-30 15:02:00 | INFO     | AdaptiveRL | Loading Monday-WorkingHours.pcap_ISCX.csv started.
2026-07-30 15:02:00 | INFO     | AdaptiveRL | Reading /home/kalpe/projects/adaptive_rl_anomaly_detection/datasets/raw/Monday-WorkingHours.pcap_ISCX.csv
2026-07-30 15:02:03 | INFO     | AdaptiveRL | Loaded Monday-WorkingHours.pcap_ISCX.csv | Shape=(529918, 79)
2026-07-30 15:02:03 | INFO     | AdaptiveRL | Loading Monday-WorkingHours.pcap_ISCX.csv completed in 2.8785 seconds.
2026-07-30 15:02:03 | INFO     | AdaptiveRL | Loading Tuesday-WorkingHours.pcap_ISCX.csv started.
2026-07-30 15:02:03 | INFO     | AdaptiveRL | Reading /home/kalpe/projects/adaptive_rl_anomaly_detection/datasets/raw/Tuesday-WorkingHours.pcap_ISCX.csv
2026-07-30 15:02:05 | INFO     | AdaptiveRL | Loaded Tuesday-WorkingHours.pcap_ISCX.csv | Shape=(445909, 79)
2026-07-30 15:02:05 | INFO     | AdaptiveRL | Loading Tuesday-WorkingHours.pcap_ISCX.csv completed in 2.1867 seconds.
2026-07-30 15:02:05 | INFO     | AdaptiveRL | Lo

(2830743, 79)


In [9]:
from src.preprocessing.cleaner import DataCleaner

cleaner = DataCleaner()

df = cleaner.clean(df)

print(df.shape)

2026-07-30 15:02:23 | INFO     | AdaptiveRL | Replacing Infinite Values started.
2026-07-30 15:02:25 | INFO     | AdaptiveRL | Replacing Infinite Values completed in 2.6037 seconds.
2026-07-30 15:02:25 | INFO     | AdaptiveRL | Removing Duplicates started.
2026-07-30 15:02:36 | INFO     | AdaptiveRL | Removed 308381 duplicate rows.
2026-07-30 15:02:36 | INFO     | AdaptiveRL | Removing Duplicates completed in 11.5469 seconds.
2026-07-30 15:02:36 | INFO     | AdaptiveRL | Removing Missing Values started.
2026-07-30 15:02:37 | INFO     | AdaptiveRL | Removed 1564 rows containing missing values.
2026-07-30 15:02:37 | INFO     | AdaptiveRL | Removing Missing Values completed in 0.7037 seconds.
2026-07-30 15:02:37 | INFO     | AdaptiveRL | Removing Constant Columns started.
2026-07-30 15:02:39 | INFO     | AdaptiveRL | Removed 8 constant columns.
2026-07-30 15:02:39 | INFO     | AdaptiveRL | Removing Constant Columns completed in 2.1373 seconds.
2026-07-30 15:02:39 | INFO     | AdaptiveRL |

(2520798, 71)


In [10]:
df.columns = df.columns.str.strip()

In [11]:
from src.preprocessing.encoder import DataEncoder

encoder = DataEncoder(target_column=" Label")

df = encoder.fit_transform(df)

print(df.dtypes["Label"])
print(df["Label"].unique()[:10])

2026-07-30 15:03:01 | INFO     | AdaptiveRL | Encoding Dataset started.
2026-07-30 15:03:01 | INFO     | AdaptiveRL | Encoding Dataset completed in 0.3521 seconds.
2026-07-30 15:03:01 | INFO     | AdaptiveRL | Encoding completed.


int64
[ 0  7 11  6  5  4  3  8 12 14]


In [12]:
from src.preprocessing.scaler import DataScaler

scaler = DataScaler(
    method="standard",
    target_column="Label",
)

df = scaler.fit_transform(df)

print(df.shape)
print(df["Label"].dtype)
print(df["Label"].unique()[:10])

2026-07-30 15:03:09 | INFO     | AdaptiveRL | Scaler (standard) fitted on 70 feature columns.
2026-07-30 15:03:10 | INFO     | AdaptiveRL | Scaling Dataset started.
2026-07-30 15:03:13 | INFO     | AdaptiveRL | Scaling Dataset completed in 2.7388 seconds.
2026-07-30 15:03:13 | INFO     | AdaptiveRL | Scaling completed.


(2520798, 71)
int64
[ 0  7 11  6  5  4  3  8 12 14]


In [13]:
X = df.drop(columns=["Label"])
y = df["Label"]

print(f"Features shape : {X.shape}")
print(f"Labels shape   : {y.shape}")

print("\nLabel Distribution:")
print(y.value_counts().sort_index())

Features shape : (2520798, 70)
Labels shape   : (2520798,)

Label Distribution:
Label
0     2095057
1        1948
2      128014
3       10286
4      172846
5        5228
6        5385
7        5931
8          11
9          36
10      90694
11       3219
12       1470
13         21
14        652
Name: count, dtype: int64


In [14]:
config = OneClassSVMConfig(
    kernel="rbf",
    gamma="scale",
    nu=0.05,
)

In [15]:
model = OneClassSVMModel(config)

model

2026-07-30 15:06:01 | INFO     | src.models.one_class_svm | One-Class SVM initialized.


OneClassSVMModel(kernel=rbf, nu=0.05, gamma=scale)

In [16]:
model.fit(X)

2026-07-30 15:06:31 | WARNING  | src.models.one_class_svm | Dataset contains 2520798 samples. Sampling 50000 rows for One-Class SVM training.
2026-07-30 15:06:31 | INFO     | src.models.one_class_svm | Training One-Class SVM...
2026-07-30 15:06:52 | INFO     | src.models.one_class_svm | Training samples: 50000 | Features: 70
2026-07-30 15:06:52 | INFO     | src.models.one_class_svm | One-Class SVM training completed successfully.


OneClassSVMModel(kernel=rbf, nu=0.05, gamma=scale)

In [17]:
X_test = X.head(5000)

predictions = model.predict(X_test)

predictions[:20]

array([0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [18]:
scores = model.anomaly_score(X_test)

scores[:20]

array([-29.55522539,  10.03274125, -24.82750501,  10.07196462,
       -43.19389039, -46.92163218, -43.62776624, -42.34101016,
       -41.27477631, -43.4792211 , -46.47870315, -42.57173858,
       -46.94292133, -42.64318304, -43.18309075, -34.72894518,
       -46.86612877, -43.57417788, -43.86520834, -42.08124225])

In [19]:
import numpy as np

unique, counts = np.unique(
    predictions,
    return_counts=True,
)

print(dict(zip(unique, counts)))

{np.int64(0): np.int64(4570), np.int64(1): np.int64(430)}


In [20]:
from pathlib import Path

save_dir = Path("trained_models")
save_dir.mkdir(exist_ok=True)

model_path = save_dir / "one_class_svm.joblib"

model.save(model_path)

print(f"Model saved to: {model_path}")

2026-07-30 15:14:09 | INFO     | src.models.one_class_svm | One-Class SVM saved -> trained_models/one_class_svm.joblib


Model saved to: trained_models/one_class_svm.joblib


In [21]:
loaded_model = OneClassSVMModel.load(model_path)

loaded_model

2026-07-30 15:14:16 | INFO     | src.models.one_class_svm | One-Class SVM loaded <- trained_models/one_class_svm.joblib


OneClassSVMModel(kernel=rbf, nu=0.05, gamma=scale)

In [22]:
pred_original = model.predict(X_test)
pred_loaded = loaded_model.predict(X_test)

print(np.array_equal(
    pred_original,
    pred_loaded,
))

True


In [23]:
scores_original = model.anomaly_score(X_test)
scores_loaded = loaded_model.anomaly_score(X_test)

print(np.allclose(
    scores_original,
    scores_loaded,
))

True


In [24]:
print("Configuration")
print(model.config)

print("\nEstimator")
print(model.estimator)

print("\nTraining Samples")
print(model.training_samples)

print("\nTraining Features")
print(model.training_features)

print("\nModel Representation")
print(model)

Configuration
OneClassSVMConfig(kernel='rbf', degree=3, gamma='scale', coef0=0.0, tol=0.001, nu=0.05, shrinking=True, cache_size=200, verbose=False, max_iter=-1, random_state=42, max_training_samples=50000)

Estimator
OneClassSVM(nu=0.05)

Training Samples
50000

Training Features
70

Model Representation
OneClassSVMModel(kernel=rbf, nu=0.05, gamma=scale)


In [25]:
print("Model Metadata")
print("-" * 50)

print(model.metadata)

Model Metadata
--------------------------------------------------
ModelMetadata(model_name='OneClassSVM', model_version='1.0.0', training_timestamp='2026-07-30T15:06:52', training_time_seconds=None, n_features=True, random_state=42, additional_info={})


In [26]:
predictions[:20]

array([0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [27]:
for p, s in zip(predictions[:20], scores[:20]):
    print(f"Prediction: {p} | Score: {s:.4f}")

Prediction: 0 | Score: -29.5552
Prediction: 1 | Score: 10.0327
Prediction: 0 | Score: -24.8275
Prediction: 1 | Score: 10.0720
Prediction: 0 | Score: -43.1939
Prediction: 0 | Score: -46.9216
Prediction: 0 | Score: -43.6278
Prediction: 0 | Score: -42.3410
Prediction: 0 | Score: -41.2748
Prediction: 0 | Score: -43.4792
Prediction: 0 | Score: -46.4787
Prediction: 0 | Score: -42.5717
Prediction: 0 | Score: -46.9429
Prediction: 0 | Score: -42.6432
Prediction: 0 | Score: -43.1831
Prediction: 0 | Score: -34.7289
Prediction: 0 | Score: -46.8661
Prediction: 0 | Score: -43.5742
Prediction: 0 | Score: -43.8652
Prediction: 0 | Score: -42.0812


In [28]:
print(scores.min())
print(scores.max())
print(scores.mean())

-72.29001536363432
100.2054123138906
-29.000522407048347
